<a href="https://colab.research.google.com/github/brceno22/LWP-virtual-assistant-/blob/main/LowPayments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# 1. SIMULACIÓN DE LA BASE DE DATOS (MOCK DATA)
data_mock = {
  "parametros_globales": {
    "nombre_empresa": "Low Weekly Payments (LWP)",
    "multiplicador_52_semanas": 2.5,
    "oferta_90_dias": "Multiplicador de 2.25 y opción de liquidación temprana al 50% del subtotal.",
    "cargo_pago_tardio_usd": 5.00,
    "fee_pago_efectivo_telefono_usd": 2.00,
    "descuento_compra_temprana_epo_pct": 40
  },
  "clientes": [
    {
      "id_cliente": "LWP-9981",
      "nombre": "Juan Perez",
      "correo": "juan.perez@example.com",
      "ssn_dni": "6789", # Guardamos solo los últimos 4 dígitos para validar
      "acuerdos": [
        {
          "producto": "PlayStation 5 Slim Bundle",
          "estado": "Activo",
          "cuota_semanal_usd": 35.00,
          "balance_total_usd": 420.00,
          "monto_vencido_overdue_usd": 0.00,
          "semanas_pagadas": 14,
          "semanas_totales_contrato": 52,
          "proximo_vencimiento": "2026-06-08"
        }
      ]
    }
  ]
}

# 2. CARGA DEL MODELO PHI-3-MINI EN 4-BITS
model_id = "microsoft/Phi-3-mini-4k-instruct"

# Configuración para que quepa en la GPU gratuita de Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Cargando modelo y tokenizador (esto puede demorar un par de minutos la primera vez)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Creamos el pipeline de generación de texto
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


# 3. LÓGICA DE FILTRADO Y ARMADO DE PROMPT
def buscar_cliente(correo, ssn_cuatro_digitos):
    for cliente in data_mock["clientes"]:
        if cliente["correo"].lower() == correo.lower() and cliente["ssn_dni"] == ssn_cuatro_digitos:
            return cliente
    return None

def consultar_asistente(mensaje_usuario, correo=None, ssn=None):
    globales = data_mock["parametros_globales"]
    cliente = buscar_cliente(correo, ssn) if (correo and ssn) else None

    # 1. INSTRUCCIONES ESTRICTAS (GUARDRAILS) + ESPAÑOL NEUTRO
    system_prompt = (
        f"Eres el asistente virtual automatizado de la empresa {globales['nombre_empresa']}.\n"
        "REGLA DE ORO 1: Debes expresarte EXCLUSIVAMENTE en ESPAÑOL NEUTRO (evita modismos de Argentina, México, España, etc. Usa 'usted' o 'tú' de forma profesional).\n"
        "REGLA DE ORO 2: Solo estás capacitado para responder preguntas sobre la empresa LWP, sus políticas generales o los datos del cliente proporcionados abajo.\n"
        "REGLA DE ORO 3: Si el usuario te pregunta sobre temas externos (deportes, cultura, geografía, celebridades, tareas escolares, etc.), debes responder estrictamente: "
        "'Lo siento, como asistente de Low Weekly Payments, solo puedo asistirle con consultas relacionadas a nuestros servicios y sus acuerdos financieros.' No intentes responder la pregunta externa.\n\n"
    )

    system_prompt += f"Políticas generales de LWP: Cargo por pago tardío: ${globales['cargo_pago_tardio_usd']}. Oferta 90 días: {globales['oferta_90_dias']}.\n"

    if cliente:
        system_prompt += (
            f"Estás atendiendo al cliente registrado: {cliente['nombre']}.\n"
            f"Datos de sus acuerdos actuales:\n{json.dumps(cliente['acuerdos'], indent=2)}\n"
            "Calcula las cuotas restantes restando las semanas pagadas de las semanas totales del contrato. Responde de forma breve y precisa."
        )
    else:
        system_prompt += (
            "El usuario actual es un VISITANTE no identificado.\n"
            "Si te pregunta por cuotas, saldos o datos personales, debes decirle de forma muy amable que para acceder a dicha información confidencial es requisito obligatorio que se identifique en la barra lateral con su correo electrónico y número de Seguro Social (SSN)."
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": mensaje_usuario}
    ]

    # Ajustamos la temperatura un poco más baja (0.2) para que sea MENOS creativo y más preciso
    outputs = pipe(messages, max_new_tokens=250, temperature=0.2, do_sample=True)
    return outputs[0]["generated_text"][-1]["content"]

    # Generación de la respuesta
    outputs = pipe(messages, max_new_tokens=250, temperature=0.2, do_sample=True)
    return outputs[0]["generated_text"][-1]["content"]

Cargando modelo y tokenizador (esto puede demorar un par de minutos la primera vez)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [ ]:
respuesta = consultar_asistente("Hola, ¿cuánto cobran si me demoro en pagar la cuota y de qué se trata la oferta de 90 días?")
print(respuesta)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


 Hola,

En Low Weekly Payments (LWP), si se retrasa el pago, se aplicará un cargo de $5.00 adicional. Es importante destacar que esta oferta de cuotas de 90 días viene acompañada por un multiplicador de 2.25, lo que significa que, en caso de liquidación temprana, podrás cobrar al 50% del subtotal.

Sin embargo, para acceder a detalles específicos como cuotas, saldos o cualquier otra información personal, nuestros usuativas deben identificarse a través de su correo electrónico y número de Seguro Social (SSN). Esto nos permite brindarles el servicio personalizado y garantizar la privacidad y seguridad de sus datos. Gracias por su comprensión y preferencia por nuestras prácticas de privacidad.


In [ ]:
respuesta = consultar_asistente("¿Me decís cuántas cuotas me quedan por favor?")
print(respuesta)

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Lo siento, pero como asistente virtual de Low Weekly Payments (LWP), no tengo acceso a informações personales ni a cuotas de pagos de individuos sin una identificación verificada. Me gustaría que me proporcionara su correo electrónico y número de Seguro Social para verificar nuestra cuenta y ayudarme a gestionar sus pagos. Solo así puedo brindarles la información que necesiten seguir la política de privacidad de la empresa.


In [ ]:
# Le pasamos el correo y los 4 dígitos correctos que están en nuestro JSON de prueba
respuesta = consultar_asistente(
    mensaje_usuario="¿Hola, cuántas cuotas me quedan del Play y cuándo vence?",
    correo="juan.perez@example.com",
    ssn="6789"
)
print(respuesta)

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Hola Juan, según tus datos actuales, quedan 38 semanas de cuotas por pagar en tu contrato de PlayStation 5 Slim Bundle. Su próximo vencimiento es el 8 de junio de 2026.


In [ ]:
respuesta = consultar_asistente("Cuantos mundiales tiene argentina")
print(respuesta)

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Lo siento, como asistente de Low Weekly Payments, solo puedo asistirle con consultas relacionadas a nuestros servicios y sus acuerdos financieros.
